### Step 1:
#### In the next code block, we import all the python packages that we need to run our code.

Python packages are like libraries in RStudio, which is a collection of code that helps us perform different activities. For example, data analysts would need some basic packages but engineers would need advanced coding hence different packages.

In [ ]:
# Step 1: Import necessary python packages

!pip install splink
from IPython.display import display

import pandas as pd
import splink
import numpy as np
import os

for d in ['linkage_results', 'charts']:
    os.makedirs(d, exist_ok=True)




### Step 2:
#### In the next code block, we import our crash and hospital dataset.

We also have a print code, that tells us the shape of our datasets: number of rows * number of columns.

In [ ]:

#Step 2: Importing crash and hospital datasets

df_crash = pd.read_csv("synthetic_crash.csv", index_col = 0)

df_hosp = pd.read_csv("synthetic_hosp.csv", index_col = 0)

#Checking number of rows * columns

print(f"\nCrash records: {df_crash.shape[0]} rows x {df_crash.shape[1]} columns")
print(f"Hospital records: {df_hosp.shape[0]} rows x {df_hosp.shape[1]} columns")


#### In the following code, we display our tables to examine each column and its values.

If you notice, there are some descripancies in column names and date formatting, gender (M/F vs Male/Female), County. We need consistent formatting in both crash and hospital dataset to link them.

#### We will address these in our data cleaning steps.

NOTE: PYTHON is case-sensitive

In [ ]:
from IPython.display import display #Enables us to view 2 datasets in the output

# View crash and hospital data

display(df_crash.head(5))
display(df_hosp.head(5))

#### Step 3: In the following code, we rename columns so both datasets share the same column names for linkage

In [ ]:
# Step 3: Rename columns so both datasets share the same column names for linkage

#Crash Columns:

df_crash.rename(columns={
    'record_id':    'unique_id',
    'first_name':   'firstname',
    'last_name':    'lastname',
    'gender':       'sex',
    'bday':         'dob',
    'crash_date':   'admit_date',
    'crash_hour':   'admit_hour',
}, inplace=True)

# Hospital Columns:

df_hosp.rename(columns={
    'patient_record_id': 'unique_id',
    'first_name':        'firstname',
    'last_name':         'lastname',
    'gender':            'sex',
    'bday':              'dob',

}, inplace=True)



### Next, we check if all the columns were renamed correctly.

In [ ]:
# Check that column names match after renaming

from IPython.display import display
display(df_crash.head(2))
display(df_hosp.head(2))


### Step 4: We check the column types, which means how the data is stored in python.

For example, in SAS we have numeric and character variables/columns

Similarly, python has:

for text values: object (default for text), string,  
numeric values: float64, int64

Best practice is to convert everything to strings or numeric types.



In [ ]:
#Step 4: Check and convert data types for each column

print(df_crash.dtypes)
print(df_hosp.dtypes)

### Step 5: We convert columns to correct types for python to handle them efficiently.

#### Explanation for these choices:
##### Text columns are converted to `string` to ensure consistent handling and optimize text operations.
##### Numeric columns are converted to `Int64` (nullable integer) for efficient mathematical operations and to properly represent missing values (NaN).
##### Dates are temporarily converted to datetime objects for standardization, then back to formatted strings (%Y-%m-%d) to ensure consistency across datasets and compatibility with Splink's custom comparison functions.

In [ ]:
# Step 5: Convert columns to correct types

string_cols = ['firstname', 'lastname', 'sex', 'dob', 'admit_date', 'county', 'role', 'state']
int_cols    = ['age', 'admit_hour', 'zip']

for col in string_cols:
    if col in df_crash.columns:
        df_crash[col] = df_crash[col].astype('string')
    if col in df_hosp.columns:
        df_hosp[col] = df_hosp[col].astype('string')

for col in int_cols:
    if col in df_crash.columns:
        df_crash[col] = pd.to_numeric(df_crash[col], errors='coerce').astype('Int64')
    if col in df_hosp.columns:
        df_hosp[col] = pd.to_numeric(df_hosp[col], errors='coerce').astype('Int64')

print(df_crash.dtypes)
print(df_hosp.dtypes)



### After running the code above, you will notice all the columns are either Strings or Int64.

### Step 6: Data Cleaning Steps

##### 1. Match Sex formatting: Male/Female from Hospital data is converted to M/F to match crash data
##### 2. Lowercase county in both the datasets: Albany/ALBANY - albany
##### 3. Changing date format to %Y-%m-%d - for admit_date and dob


#### Step 7: Adding source markers

##### We add a column source (crash/hosp) and add a marker for column unique_id so that we know where these columns came from after linkage. It is good practice for quality check



In [ ]:
# Step 6
#Match crash and hosp sex

df_hosp['sex'] = df_hosp['sex'].astype('string').replace({'Male': 'M', 'Female': 'F'})

#lowercase 'county'

for _df in (df_crash, df_hosp):
    _df['county'] = _df['county'].str.lower()

#Standardize date formats
df_crash['admit_date'] = pd.to_datetime(df_crash['admit_date'], errors='coerce').dt.strftime('%Y-%m-%d').astype('string')
df_hosp['admit_date']  = pd.to_datetime(df_hosp['admit_date'],  errors='coerce').dt.strftime('%Y-%m-%d').astype('string')

df_crash['dob'] = pd.to_datetime(df_crash['dob'], errors='coerce').dt.strftime('%Y-%m-%d').astype('string')
df_hosp['dob']  = pd.to_datetime(df_hosp['dob'],  errors='coerce').dt.strftime('%Y-%m-%d').astype('string')

# %%
# Step 7: Add source markers
df_crash['unique_id'] = 'crash_' + df_crash.index.astype(str)
df_hosp['unique_id']  = 'hosp_'  + df_hosp.index.astype(str)

df_crash['source'] = 'crash'
df_hosp['source']  = 'hosp'

# Verify that all columns are now numeric or string, and match between datasets
df_crash.dtypes
df_hosp.dtypes

### In the next steps (8 /9), we generate charts:

#### 1. Completeness Chart:
A bar chart that shows how complete (non-missing) each column is across your dataset. For each field like name, date of birth, or county, it tells you what percentage of records actually have a value. This is useful for identifying columns with too much missing data before you start linking a column that is mostly empty is not a reliable field to match on.

#### 2. Profile Columns Chart
A visual summary of the distribution of values in each column. For example, it shows how many unique values exist, which values appear most frequently, and how evenly spread the data is. This helps you understand your data before building your linkage model for instance, a column where everyone has the same value (like State in our data) is not useful for distinguishing between records.


In [ ]:
# Step 8: Import splink modules
from splink import DuckDBAPI
from splink.exploratory import completeness_chart
from splink.exploratory import profile_columns

# Step 9: Generate and inspect charts to understand your data

# Completeness Chart -- check data missingness and completeness
completeness_chart(
    [df_crash, df_hosp],
    db_api = DuckDBAPI(),
    table_names_for_chart = ["crash", "hosp"]).save("charts/completeness_chart.html")

# Profile Columns Charts -- shows distributions of values for each variable
profile_columns(df_crash, db_api = DuckDBAPI()).save("charts/profile_columns_crash.html")
profile_columns(df_hosp, db_api = DuckDBAPI()).save("charts/profile_columns_hcup.html")



#### Step 10: Define Blocking Rules

Instead of comparing every record in the crash dataset to every record in the hospital dataset, blocking rules narrow down the comparisons to only record pairs that share a common value like the same last name or date of birth. This makes the linkage process faster and more efficient.

Here we define five blocking rules and generate a **cumulative comparisons chart** to visualize how many record pairs each rule produces. This helps us assess whether our blocking rules are capturing enough potential matches before we build the linkage model.

In [ ]:
# Step 10: Define Blocking Rules

# Import Splink modules
from splink import block_on
from splink.blocking_analysis import cumulative_comparisons_to_be_scored_from_blocking_rules_chart
import splink.comparison_library as cl
from splink import DuckDBAPI, Linker, SettingsCreator, block_on
db_api = DuckDBAPI()

# Here the we use blocking rules to assess the number of matches a combination of columns will produce
#which is used in the next step of generating cumulative comparisons chart

blocking_rules = [
    block_on('lastname', 'age'),
    block_on('dob', 'age', 'sex'),
    block_on('dob', 'county'),
    block_on('firstname', 'county'),
    block_on('firstname', 'lastname')
]
# Cumulative comparisons chart -- used to assess your blocking rules


cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables = [df_crash, df_hosp],
    blocking_rules = blocking_rules,
    db_api = db_api,
    link_type = 'link_only',
    unique_id_column_name = 'unique_id',

    source_dataset_column_name = 'source').save("charts/cumulative_comparisons.html")

### You will see blocking rules at 3 different sites:
#### 1. To assess number of matches a combination of columns will produce: this will help to select your model training blocks.
#### 2. Pre-training: Where you ask Splink to use random samples to link. This helps us to learn pre-training estimates (lambda and u-values)
#### 3. Session training aka EM-training: Here you train your model and Splink learns about the match weight and probability for all the rows in your data.

### Step 11: Custom Comparisons for Admit Date and Admit Hour

A person injured in a crash may not be admitted to the hospital on the same day. There can be a delay of hours or even days.

To account for this, we define two **custom comparisons**:

- **Admit Date** — allows up to a 2-day difference between crash date and hospital admission date
- **Admit Hour** — allows up to a 48-hour difference when combining date and hour together

Each comparison has four levels that Splink evaluates in order — null, exact match, within the time window, and everything else.

This gives the model a more nuanced picture of how well two records align on timing, rather than a simple yes or no.

> 💡 You can customize this based on your needs.

> 💡 You don't need to memorize this code, just know that it tells Splink *"a match doesn't have to be exact, just close enough in time to be realistic."*

In [ ]:
# Step 11: Define custom comparisons for admit date and admit hour

from splink.comparison_library import CustomComparison

admit_date_comp = CustomComparison(
    output_column_name='admit_date_comp',
    comparison_levels=[
        {
            'sql_condition': 'admit_date_r IS NULL OR admit_date_l IS NULL',
            'label_for_charts': 'null',
            'is_null_level': True
        },
        {
            'sql_condition': (
                'admit_date_l IS NOT NULL AND admit_date_r IS NOT NULL AND '
                'CAST(admit_date_l AS DATE) = CAST(admit_date_r AS DATE)'
            ),
            'label_for_charts': 'Exact match on admit_date',
        },
        {
            'sql_condition': (
                "ABS(DATE_DIFF('day', CAST(admit_date_r AS DATE), CAST(admit_date_l AS DATE))) <= 2"
            ),
            'label_for_charts': 'admit_date within 2 days',
        },
        {
            'sql_condition': 'TRUE',
            'label_for_charts': 'all other comparisons'
        }
    ]
)

# Elapsed time between crash and admit: uses date + hour (same 48h rule as before; first match wins).
admit_hour_comp = CustomComparison(
    output_column_name='admit_hour_comp',
    comparison_levels=[
        {
            'sql_condition': (
                'admit_date_r IS NULL OR admit_date_l IS NULL OR '
                'admit_hour_r IS NULL OR admit_hour_l IS NULL'
            ),
            'label_for_charts': 'null',
            'is_null_level': True
        },
        {
            'sql_condition': (
                'admit_date_l IS NOT NULL AND admit_date_r IS NOT NULL AND '
                'admit_hour_l IS NOT NULL AND admit_hour_r IS NOT NULL AND '
                'CAST(admit_date_l AS DATE) = CAST(admit_date_r AS DATE) AND '
                'CAST(admit_hour_l AS DOUBLE) = CAST(admit_hour_r AS DOUBLE)'
            ),
            # Same idea as ExactMatch, but needs BOTH date and hour (hour alone is misleading across days).
            'label_for_charts': 'Exact match (date and hour)',
        },
        {
            'sql_condition': """
                ABS(
                    ((CAST(admit_date_r AS DATE) - CAST(admit_date_l AS DATE)) * 24)
                    + (CAST(admit_hour_r AS FLOAT) - CAST(admit_hour_l AS FLOAT))
                ) <= 48
            """,
            'label_for_charts': 'Within 48 hours (combined date and hour)',
        },
        {
            'sql_condition': 'TRUE',
            'label_for_charts': 'all other comparisons'
        }
    ]
)

### Step 12: Define Comparisons

Here we choose which fields to compare between the crash and hospital records, and how to compare them. For most fields we use **ExactMatch**, meaning both records must have the same value to score a point. For admit date and admit hour we plug in the **custom comparisons** we built in Step 11.

The choice of comparison method depends on how rich and reliable your data is:

- Fields like `sex`, `dob`, and `county` are straightforward — exact match works well
- Name fields like `firstname` and `lastname` could use fuzzy methods like **Jaro-Winkler** or **Levenshtein** to account for typos and spelling variations.

> 💡 The comparisons you choose directly shape the quality of your linkage and picking the right method for each field is one of the most important decisions in the process.

In [ ]:
# Step 12: Choose comparison methods you wish to use (e.g. ExactMatch)

comparisons = [
        cl.ExactMatch('firstname'),
        cl.ExactMatch('lastname'),
        cl.ExactMatch('sex'),
        cl.ExactMatch('dob'),
        cl.ExactMatch('age'),
        cl.ExactMatch('county'),


        admit_date_comp, #Using the custom comparison we created in Step 10
        admit_hour_comp,
    ]

### Steps 13–15: Assemble the Model and Pre-Training

Here we bring all our previous steps together: the blocking rules and comparisons and hand them to Splink as a complete set of instructions.

Before Splink can start linking, it needs to learn two things from your data:
- **How many true matches to roughly expect** (Step 14)
- **How often fields like name or date of birth match purely by chance** (Step 15)

> 💡 Think of this as Splink doing its homework before the actual test. No linkage has happened yet we are just helping the model understand the data so it can make smarter decisions when matching begins.

In [ ]:

# Step 13: Define full model settings
model_settings = SettingsCreator(
    unique_id_column_name = 'unique_id',
    link_type='link_only',
    blocking_rules_to_generate_predictions=blocking_rules,
    comparisons=comparisons,
    retain_intermediate_calculation_columns=True,
)

linker = Linker(
    [df_crash, df_hosp],
    model_settings,
    db_api = DuckDBAPI()
)

#### PRE-TRAINING

# Step 14: Estimate lambda (probability two random records match)

deterministic_rules = [
    block_on('firstname', 'county'),
    block_on('firstname', 'lastname'),
    block_on('lastname', 'age'),
]

linker.training.estimate_probability_two_random_records_match(
    deterministic_rules, recall=0.8
)

# Step 15: Estimate u probabilities using random sampling

linker.training.estimate_u_using_random_sampling(max_pairs=1e7)


### Step 16: EM Training — Learning Match Weights

Now Splink starts learning from the data using the **Expectation Maximisation (EM)** algorithm. This is where it figures out how much weight to give each field when deciding if two records are a match.

Each training session blocks on a different combination of columns. The columns used for blocking in that session are treated as "assumed matches" so Splink calculates match weights for all the *other* columns instead. This is why we run multiple sessions to make sure every field gets its own match weight calculated.

> 💡 Think of it like studying for an exam by focusing on different topics each session by the end, everything is covered.

We chose these blocking combinations based on the cumulative comparisons chart we generated earlier.

###Session 1

In [ ]:
# Step 16: EM Training — estimate m probabilities

session_1 = linker.training.estimate_parameters_using_expectation_maximisation(

block_on('lastname', 'age'), estimate_without_term_frequencies=True

)


### Session 2

We repeat the training with a different blocking combination. This time `firstname` and `county` so that the remaining columns that were not covered in Session 1 get their match weights calculated.

By the end of both sessions, every field has an m-probability and the model has **converged**, meaning the values have stabilized and stopped changing significantly. Splink is now ready to start linking.

In [ ]:
session_2 = linker.training.estimate_parameters_using_expectation_maximisation(

block_on('firstname', 'county'), estimate_without_term_frequencies=True

)

### NOTE: The synthetic data only needed 2 sessions for Splink to learn all the values. But in real-world, larger data sets may require more sessions.

Eg. NY state requires 4-5 sessions

### Next, assess your model using charts

### Match Weights Chart
This chart shows how much each field **contributes to the final match decision**. Fields that are highly distinctive like date of birth will have a large positive weight, meaning they strongly increase the probability of a match when they agree. Fields that are less informative will have smaller weights like sex.

> 💡 This is a great way to sanity check your model if a field you expected to be important has a very low weight, it may be a sign of poor data quality or that the field needs a different comparison method.

> For example, `admit_hour` with a simple exact match comparison would have a near-zero weight because a crash and hospital admission rarely happen on the exact same hour. But it still is an important column that should contribute to our model hence the custom comparisons.

This chart helps you to examine this.

In [ ]:
linker.visualisations.match_weights_chart()


### Waterfall Chart

While the match weights chart summarizes how the model behaves **overall**, the waterfall chart zooms in on **one specific pair of records** and shows exactly how Splink arrived at its final match score.

Each bar represents a single field like name, date of birth, and shows whether that field pushed the score **up** (evidence for a match) or **down** (evidence against a match). It starts from the baseline probability and builds up or down from there.

> 💡 This is one of the most useful charts for spot-checking your results if two records were linked that shouldn't be, or weren't linked when they should be, the waterfall chart tells you exactly why.

In [ ]:
# WATERFALL CHART

df_predictions = linker.inference.predict()
records_to_view = df_predictions.as_pandas_dataframe().sample(3).to_dict(orient='records')
linker.visualisations.waterfall_chart(records_to_view, filter_nulls=False)


### Unlinkables Chart

This chart answers a simple but important question: **how many records in your data are too incomplete to ever find a match?**

Splink tests each record by trying to match it against itself. If a record cannot even match itself, it lacks enough information to be linked to anything. The chart shows what percentage of records fall into this category at different match thresholds.

A low percentage means your data is in good shape. In our synthetic data:
- At a relaxed threshold of **0.3** — only 0.05% of records are unlinkable ✅
- At a very strict threshold of **0.99998** — 7.85% are unlinkable, which is expected

> 💡 This chart helps you choose a realistic threshold for your final predictions. The stricter you are, the more records get left out, so it is about finding the right balance for your research question.

In [ ]:
#Unlinkables chart

linker.evaluation.unlinkables_chart()

### Step 17: Run the Linkage

This is the step where the actual linking happens. Splink takes everything it has learned: the blocking rules, comparisons, and match weights and applies it to generate a **match probability** for every candidate record pair.

The result is a table of record pairs, each with a probability score between 0 and 1 indicating how likely they are to be the same person. We preview the first 5 rows to take a quick look at the output.

> 💡 All the previous steps were preparation. This is the moment Splink puts it all together and makes its predictions.

In [ ]:
#Step 17: Run the linkage

df_predictions = linker.inference.predict()
df_predictions.as_pandas_dataframe(limit=5)

### Step 18: Prepare Results for Export

We select the columns we want to keep from the predictions including the match probability and weight, identifiers from both datasets, and the key fields we compared.

We then convert the results into a format (pandas dataframe) that can be easily filtered, reviewed, or saved as a spreadsheet.

The `_l` and `_r` suffixes indicate which dataset each column comes from — **l** for the crash (left) dataset and **r** for the hospital (right) dataset.

In [ ]:
# Step 18: STEPS FOR EXPORTING:

#COLUMNS WE WANT TO EXPORT

cols_keep = [
    'unique_id_l', 'unique_id_r',
    'match_probability', 'match_weight', 'match_key',
    'firstname_l', 'firstname_r','lastname_l',  'lastname_r',
    'dob_l', 'dob_r', 'sex_l',  'sex_r',  'age_l', 'age_r',

    'county_l', 'county_r'
    'role_l', 'role_r',
    'admit_date_l', 'admit_date_r',
    'admit_hour_l', 'admit_hour_r',
]

df_predictions_pd = df_predictions.as_pandas_dataframe()

### Filtering and Exporting Results

We filter the predictions to keep only record pairs with a **match probability of 80% or higher** meaning Splink is at least 80% confident these two records belong to the same person.

> 💡 The 0.8 threshold is a starting point. You can raise it to be more strict (fewer but higher confidence matches) or lower it to cast a wider net.

The right threshold depends on your research question and your data.

In [ ]:
#Can filter based on match probability and export as .csv

df_filtered = df_predictions_pd[df_predictions_pd['match_probability'] >= 0.8]

df_filtered = df_filtered[[c for c in cols_keep if c in df_filtered.columns]]

df_filtered.to_csv('linkage_results/linkage_results_filtered.csv', index=False)

print(f"Matched records saved: {len(df_filtered)} rows")

### Exporting All Results

This exports the **complete set of predictions** regardless of match probability including low confidence pairs.

> 💡 This is useful if you want to review the full picture, apply a different threshold later, or do further analysis on borderline matches.

In [ ]:
#EXPORTS YOUR ENTIRE LINKAGE (irrespective of match probability)

df_predictions_export = df_predictions_pd [[c for c in cols_keep if c in df_predictions_pd.columns]]

df_predictions_export.to_csv('linkage_results/linkage_results.csv', index=False)
print(f"Matched records saved: {len(df_predictions_export)} rows")